# Astra 200M — V1 Distillation Notebook (Gemini Flash-Lite)

Generates a robust, structured cybersecurity training dataset using **Gemini Flash-Lite** (`google/gemini-flash-lite-latest`) for teacher distillation, targeting anomaly detection, early-warning sequences, false positives, and multi-signal correlation.

**Budget / Distribution:**
- 20% Normal behavior
- 20% Suspicious behavior
- 20% Early-warning sequences
- 10% False positives
- 10% Multi-signal correlation
- 5% Authentication
- 5% Network
- 5% Process behavior
- 5% Vulnerability / code security

Outputs: `/content/astra/datasets/distillation/astra_v1_distillation.jsonl`

In [ ]:
!pip install -q google-genai pydantic
import os, json, time, random
from google import genai
from google.genai import types

api_key = os.environ.get('TEACHER_API_KEY')
if not api_key:
    from google.colab import userdata
    try:
        api_key = userdata.get('TEACHER_API_KEY')
    except Exception:
        pass

assert api_key, 'Please set TEACHER_API_KEY in environment variables or Colab Secrets.'
client = genai.Client(api_key=api_key)
print('Gemini client initialized successfully.')

In [ ]:
import os
os.makedirs('datasets/distillation', exist_ok=True)

SYSTEM_PROMPT = """You are the teacher model for Astra 200M, a cybersecurity research model being trained for defensive security analysis.
Your job is to generate high-quality cybersecurity training examples that help a smaller student model learn:
- anomaly detection
- suspicious behavior detection
- early security warnings
- security-event sequence analysis
- vulnerability identification
- risk assessment
- evidence-based reasoning
- defensive recommendations

Focus on defensive cybersecurity. Do not provide instructions for unauthorized exploitation.
Return ONLY valid JSON with keys: input, classification (normal|suspicious|malicious|unknown), risk (info|low|medium|high|critical), early_warning (bool), evidence (list of strings), analysis (string), recommended_action (string), confidence (float 0.0-1.0)."""

CATEGORIES = [
    ("Normal behavior", 0.20, "Generate a realistic example of completely normal Linux authentication activity or system telemetry."),
    ("Suspicious behavior", 0.20, "Generate a realistic cybersecurity scenario containing subtle suspicious behavior not yet confirming an attack."),
    ("Early-warning sequences", 0.20, "Generate a realistic chronological cybersecurity event sequence containing 5-15 events starting normal and introducing subtle early warning anomalies."),
    ("False positives", 0.10, "Generate a realistic cybersecurity scenario that initially looks suspicious but has a legitimate explanation."),
    ("Multi-signal correlation", 0.10, "Generate a cybersecurity scenario where no individual event is malicious, but combined events create a warning."),
    ("Authentication", 0.05, "Generate a cybersecurity authentication scenario with mixed normal/unusual/suspicious patterns."),
    ("Network", 0.05, "Generate a realistic network-security event sequence involving DNS, connections, and traffic anomalies."),
    ("Process behavior", 0.05, "Generate a system-process event sequence requiring contextual reasoning."),
    ("Vulnerability / code security", 0.05, "Generate a secure/insecure code or configuration scenario focusing on defensive detection.")
]

print('Setup complete. Ready to generate dataset.')

In [ ]:
OUTPUT_PATH = 'datasets/distillation/astra_v1_distillation.jsonl'
TARGET_SAMPLES = 1000  # Adjust as needed (e.g. 5000-20000)

generated_count = 0
with open(OUTPUT_PATH, 'w', encoding='utf-8') as f:
    while generated_count < TARGET_SAMPLES:
        r = random.random()
        cum = 0
        cat_name, cat_weight, cat_prompt = CATEGORIES[0]
        for name, weight, prompt in CATEGORIES:
            cum += weight
            if r <= cum:
                cat_name, cat_weight, cat_prompt = name, weight, prompt
                break
        
        full_prompt = f"{cat_prompt}\nReturn ONLY valid JSON matching the specified schema."
        try:
            response = client.models.generate_content(
                model='google/gemini-flash-lite-latest',
                contents=full_prompt,
                config=types.GenerateContentConfig(
                    system_instruction=SYSTEM_PROMPT,
                    response_mime_type="application/json",
                    temperature=0.7,
                )
            )
            text = response.text.strip()
            data = json.loads(text)
            data["source"] = "gemini-flash-lite-distillation"
            data["category"] = cat_name
            data["synthetic"] = True
            
            f.write(json.dumps(data) + '\n')
            generated_count += 1
            if generated_count % 50 == 0:
                print(f'Generated {generated_count}/{TARGET_SAMPLES} samples...')
        except Exception as e:
            print(f'Error generating sample: {e}')
            time.sleep(1.0)

print(f'Distillation complete! Saved to {OUTPUT_PATH}')